# 34 — Explainable AI and Interpretability — Lab

## Description

This lab applies three core explanation techniques to real data: SHAP to a credit-scoring model, LIME to an image classifier, and stability analysis to evaluate explanation robustness. Students generate attributions, interpret visualizations, and measure how perturbations affect explanations. This hands-on work reinforces the lecture's concepts of local attribution, neighborhood sampling, and explanation fragility.

## Objectives

- Student can generate SHAP waterfall and summary plots for a tabular model using `shap.KernelExplainer` and interpret the top three positive and negative attributions.
- Student can generate LIME superpixel explanations for an image classifier and identify the five most influential superpixels for a given prediction.
- Student can measure explanation stability as the standard deviation of SHAP values across 20 perturbed copies of a single input and report whether stability is above or below a defined threshold.

## Prerequisites

- Complete the lecture content for m34, particularly sections on SHAP, LIME, and explanation stability.
- Familiarity with neural networks and gradient computation from m03 (required for Integrated Gradients implementation).
- Basic proficiency with numpy, scikit-learn, and pandas.
- Knowledge of image classification and convolutional networks from m04.

## Required Software/Packages

- `shap` (version 0.41.0 or later) — for KernelSHAP and SHAP visualizations.
- `lime` (version 0.2.0 or later) — for LIME explanations.
- `torch` and `torchvision` (PyTorch, latest stable) — for deep network definition and image loading.
- `captum` (version 0.5.0 or later) — for Integrated Gradients.
- `scikit-learn` (version 1.0 or later) — for model training and preprocessing.
- `matplotlib` and `numpy` — for visualization and numerical operations.
- `requests` — for downloading model weights and datasets.

## Environment/Setup Instructions

Run the cell below to install required packages (may take 2-3 minutes).

In [ ]:
import subprocess
import sys

packages = ['shap', 'lime', 'torch', 'torchvision', 'captum']
for package in packages:
    try:
        __import__(package)
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\nAll required packages installed successfully.")

## Background/Context

Machine learning models make predictions, but explaining why a model chose a particular output is essential for accountability, debugging, and regulatory compliance. This lab walks through three explanation techniques:

1. **SHAP on tabular data**: We use a pre-trained credit-scoring model and generate feature attributions. Positive attributions push the score toward approval; negative attributions push toward denial. Visualizations show which features matter most for individual predictions and across the dataset.

2. **LIME on image data**: We apply LIME to an image classifier, segmenting images into superpixels and showing which regions influence the prediction. This is model-agnostic: LIME works on any classifier without access to weights.

3. **Explanation stability**: We perturb a test input and recompute explanations, measuring how much attributions fluctuate. High variance suggests explanations are unreliable for auditing.

Throughout, we follow the lecture framework: local explanations for individual predictions, global insights from aggregated attributions, and formal stability measurement.

## Exercise Instructions

1. Load a tabular binary-classification dataset (breast cancer dataset as a proxy for credit scoring), train a Random Forest, and apply shap.KernelExplainer to generate attributions for a test instance; produce a waterfall plot and a summary plot over 50 test instances; list the top three positive and top three negative features by magnitude. (9 min)

2. Train a small CNN image classifier on a CIFAR-10 subset and apply LIME to two test images; generate superpixel segmentation and superpixel explanations; identify the five most influential superpixels for each image. (9 min)

3. For five test instances from the tabular model, perturb each instance by adding Gaussian noise twenty times; generate SHAP values for each perturbed copy; compute the per-feature standard deviation of SHAP values across all 20 copies. (6 min)

4. Assemble a table with Feature, Mean SHAP, and Std Dev SHAP for each feature across the five test instances; flag any feature whose standard deviation exceeds 10% of its mean absolute SHAP value as unstable; report the count of unstable features. (6 min)

## Starter Code

Below is scaffolded code to get you started. Complete the marked sections.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import shap
import lime
import lime.lime_tabular
import lime.lime_image
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

### Step 1: SHAP on Tabular Credit-Scoring Data

Load a tabular binary-classification dataset (the breast cancer dataset, used here as a self-contained proxy for credit scoring: approve/deny ~ benign/malignant) and train a Random Forest. Apply KernelSHAP to generate local and global explanations.

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

# Evaluate
train_acc = model.score(X_train_scaled, y_train)
test_acc = model.score(X_test_scaled, y_test)
print(f"Model accuracy: train={train_acc:.3f}, test={test_acc:.3f}")

In [ ]:
# Apply KernelSHAP for local explanation
explainer = shap.KernelExplainer(
    model.predict_proba,
    shap.sample(X_train_scaled, 100)
)

# Instance 8 is a borderline prediction with both positive and negative
# attributions, so top-3 positive AND top-3 negative feature lists are non-empty
test_instance = X_test_scaled[8:9]
shap_values = explainer.shap_values(test_instance)

# shap >= 0.45 returns one array of shape (n_samples, n_features, n_classes);
# older versions return a list with one (n_samples, n_features) array per class.
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_class1 = shap_values[0, :, 1]
else:
    shap_values_class1 = shap_values[1][0]

print("SHAP values for test instance 8 (class 1):")
print(shap_values_class1)
print(f"\nPrediction probability: {model.predict_proba(test_instance)[0][1]:.3f}")

In [ ]:
# Create SHAP explanation object for visualization
explainer_np = shap.KernelExplainer(
    lambda x: model.predict_proba(x)[:, 1],
    shap.sample(X_train_scaled, 100)
)

shap_val = explainer_np.shap_values(test_instance)
base_value = explainer_np.expected_value

# Display top features by absolute attribution
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'SHAP': shap_val[0],
    'Abs_SHAP': np.abs(shap_val[0])
}).sort_values('Abs_SHAP', ascending=False)

print("Top 6 features by absolute SHAP value for instance 8:")
print(feature_importance.head(6))
print(f"\nTop 3 positive (push toward class 1): {feature_importance[feature_importance['SHAP'] > 0].head(3)['Feature'].tolist()}")
print(f"Top 3 negative (push toward class 0): {feature_importance[feature_importance['SHAP'] < 0].head(3)['Feature'].tolist()}")

# Local accuracy check: base value + sum of SHAP values ~ predicted probability
print(f"\nbase_value + sum(SHAP) = {base_value + shap_val[0].sum():.3f} "
      f"(predicted probability = {model.predict_proba(test_instance)[0][1]:.3f})")

# Waterfall plot for this instance
shap_explanation = shap.Explanation(
    values=shap_val[0],
    base_values=base_value,
    data=test_instance[0],
    feature_names=list(X.columns),
)
shap.plots.waterfall(shap_explanation, max_display=10)
plt.tight_layout()
plt.show()

### Expected Outputs for Step 1

The SHAP values should be a 30-dimensional array. The sum of SHAP values plus the base value should approximately equal the model's predicted probability for class 1. Top 3 positive and top 3 negative features should be clearly identified.

In [ ]:
# Summary plot: global perspective across many test instances
shap_values_global = explainer_np.shap_values(X_test_scaled[:50])

# Create summary data
summary_df = pd.DataFrame({
    'Feature': X.columns,
    'Mean_Abs_SHAP': np.abs(shap_values_global).mean(axis=0)
}).sort_values('Mean_Abs_SHAP', ascending=False)

print("Summary of SHAP values across 50 test instances:")
print(summary_df.head(10))

# Beeswarm summary plot
shap.summary_plot(
    shap_values_global,
    X_test_scaled[:50],
    feature_names=list(X.columns),
    max_display=10,
)
plt.show()

### Step 2: LIME on Image Classifier

Train a small CNN on a CIFAR-10 subset (torchvision's pre-trained models predict ImageNet classes, so a quick local training run is the simplest way to get a CIFAR-10 classifier) and apply LIME to explain its predictions using superpixel segmentation.

In [ ]:
# Load CIFAR-10 subset from the course GitHub repo

import os
import urllib.request
from torch.utils.data import DataLoader, Subset, TensorDataset

os.makedirs("cifar10_subset", exist_ok=True)

# Download the subset files (skip if already downloaded)
base_url = "https://raw.githubusercontent.com/Abuhamad/comp378_f26/main/datasets/cifar10_subset"
for fname in ["train.pt", "test.pt"]:
    dest = os.path.join("cifar10_subset", fname)
    if not os.path.exists(dest):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(f"{base_url}/{fname}", dest)

# Load datasets
train_data = torch.load("cifar10_subset/train.pt", weights_only=True)  # 5000 train images
test_data = torch.load("cifar10_subset/test.pt", weights_only=True)    # 1000 test images

train_dataset = TensorDataset(train_data["images"].permute(0, 3, 1, 2).float() / 255.0, train_data["labels"])
test_dataset = TensorDataset(test_data["images"].permute(0, 3, 1, 2).float() / 255.0, test_data["labels"])

# Subsets for quick training and evaluation
trainset_subset = Subset(train_dataset, range(500))
testset_subset = Subset(test_dataset, range(100))

train_loader = DataLoader(trainset_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset_subset, batch_size=50, shuffle=False)

# Load a batch of test images
images, labels = next(iter(test_loader))

# Convert to numpy (NHWC, values in [0, 1]) for LIME
images_np = images.numpy().transpose(0, 2, 3, 1)
true_labels = labels.numpy()

print(f"Loaded {len(images)} CIFAR-10 images for explanation")
print(f"Image shape per sample: {images_np[0].shape}")

In [ ]:
# Define and train a small CNN on the CIFAR-10 subset
# (takes under a minute on a Colab CPU/GPU)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_img = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x32 -> 16x16
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16x16 -> 8x8
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model_img.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

n_epochs = 10
model_img.train()
for epoch in range(n_epochs):
    total_loss, correct, seen = 0.0, 0, 0
    for batch_images, batch_labels in train_loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        outputs = model_img(batch_images)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_labels)
        correct += (outputs.argmax(dim=1) == batch_labels).sum().item()
        seen += len(batch_labels)
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{n_epochs}: loss={total_loss / seen:.3f}, train acc={correct / seen:.3f}")

model_img.eval()

# Quick evaluation on the held-out test subset
correct, seen = 0, 0
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        outputs = model_img(batch_images)
        correct += (outputs.argmax(dim=1) == batch_labels).sum().item()
        seen += len(batch_labels)
print(f"Test accuracy on subset: {correct / seen:.3f}")

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Prediction function for LIME: expects NHWC numpy images in [0, 1]
def predict_fn(images_batch):
    images_tensor = torch.tensor(images_batch, dtype=torch.float32).permute(0, 3, 1, 2).to(device)
    with torch.no_grad():
        outputs = model_img(images_tensor)
    return torch.softmax(outputs, dim=1).cpu().numpy()

# Verify predictions on the loaded images
preds = predict_fn(images_np)
pred_classes = np.argmax(preds, axis=1)
print(f"Model predictions (first 10): {[class_names[c] for c in pred_classes[:10]]}")
print(f"True labels       (first 10): {[class_names[c] for c in true_labels[:10]]}")

In [ ]:
# Apply LIME to the first image
from skimage.segmentation import felzenszwalb, mark_boundaries

explainer_lime = lime.lime_image.LimeImageExplainer()

explanation = explainer_lime.explain_instance(
    images_np[0],
    predict_fn,
    top_labels=5,
    num_samples=500,
    segmentation_fn=felzenszwalb
)

# Get the top label (predicted class)
top_label = explanation.top_labels[0]
print(f"Predicted class: {class_names[top_label]}")
print(f"Prediction confidence: {preds[0, top_label]:.3f}")

# Get the LIME coefficients for this label
weights = dict(explanation.local_exp[top_label])
print(f"\nTop 5 most influential superpixels (by absolute coefficient):")
sorted_weights = sorted(weights.items(), key=lambda x: abs(x[1]), reverse=True)
for superpixel_id, weight in sorted_weights[:5]:
    print(f"  Superpixel {superpixel_id}: coefficient = {weight:.4f}")

# Visualize the top 5 superpixels overlaid on the image
temp_img, mask = explanation.get_image_and_mask(
    top_label, positive_only=False, num_features=5, hide_rest=False
)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(images_np[0])
axes[0].set_title(f"Original (true: {class_names[true_labels[0]]})")
axes[1].imshow(mark_boundaries(temp_img, mask))
axes[1].set_title(f"LIME top-5 superpixels\n(pred: {class_names[top_label]})")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Expected Outputs for Step 2

LIME should identify 5 superpixels with the largest absolute weights. These superpixels either support the prediction (positive weight) or oppose it (negative weight). The output is a list of superpixel_id and coefficient pairs.

In [ ]:
# Apply LIME to the second image
explanation2 = explainer_lime.explain_instance(
    images_np[1],
    predict_fn,
    top_labels=5,
    num_samples=500,
    segmentation_fn=felzenszwalb
)

top_label2 = explanation2.top_labels[0]
print(f"Predicted class: {class_names[top_label2]}")
print(f"Prediction confidence: {preds[1, top_label2]:.3f}")

weights2 = dict(explanation2.local_exp[top_label2])
print(f"\nTop 5 most influential superpixels:")
sorted_weights2 = sorted(weights2.items(), key=lambda x: abs(x[1]), reverse=True)
for superpixel_id, weight in sorted_weights2[:5]:
    print(f"  Superpixel {superpixel_id}: coefficient = {weight:.4f}")

# Visualize the top 5 superpixels overlaid on the image
temp_img2, mask2 = explanation2.get_image_and_mask(
    top_label2, positive_only=False, num_features=5, hide_rest=False
)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(images_np[1])
axes[0].set_title(f"Original (true: {class_names[true_labels[1]]})")
axes[1].imshow(mark_boundaries(temp_img2, mask2))
axes[1].set_title(f"LIME top-5 superpixels\n(pred: {class_names[top_label2]})")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Step 3: Explanation Stability via Perturbation

Generate SHAP explanations for multiple perturbed versions of five test instances and measure stability through per-feature standard deviation.

In [ ]:
# Select 5 test instances for stability analysis
test_indices = [0, 1, 2, 3, 4]
test_instances = X_test_scaled[test_indices]

n_perturbations = 20
noise_std = 0.01

np.random.seed(42)  # reproducible perturbations

# Initialize storage for SHAP values across perturbations
shap_values_by_perturbation = {i: [] for i in test_indices}

print("Computing SHAP values for perturbed instances...")
for pos, instance_idx in enumerate(test_indices):
    original = test_instances[pos:pos + 1]

    for pert in range(n_perturbations):
        perturbed = original + np.random.normal(0, noise_std, original.shape)
        # nsamples=200 keeps the 5 x 20 = 100 KernelSHAP runs fast;
        # increase toward the default for more accurate attributions
        shap_val = explainer_np.shap_values(perturbed, nsamples=200)
        shap_values_by_perturbation[instance_idx].append(shap_val[0])

    if (pos + 1) % 2 == 0:
        print(f"  Completed {pos + 1}/5 instances")

print("Perturbation analysis complete.")

In [ ]:
# Compute per-feature standard deviation
stability_results = []

for instance_idx in test_indices:
    shap_array = np.array(shap_values_by_perturbation[instance_idx])
    mean_shap = shap_array.mean(axis=0)
    std_shap = shap_array.std(axis=0)
    
    for feat_idx, feat_name in enumerate(X.columns):
        stability_results.append({
            'Instance': instance_idx,
            'Feature': feat_name,
            'Mean_SHAP': mean_shap[feat_idx],
            'Abs_Mean_SHAP': abs(mean_shap[feat_idx]),
            'Std_SHAP': std_shap[feat_idx]
        })

stability_df = pd.DataFrame(stability_results)
print("Stability data computed for all 5 instances and all features.")
print(f"Total rows: {len(stability_df)}")

### Expected Outputs for Step 3

The output is a dataframe with columns: Instance, Feature, Mean_SHAP, Abs_Mean_SHAP, Std_SHAP. Each row represents one feature for one test instance, showing the average SHAP value and its standard deviation across 20 perturbations. The data is used in Step 4 for the summary table and instability flagging.

### Step 4: Stability Summary Table and Flagging

Aggregate stability results into a summary table and identify features where standard deviation exceeds 10% of mean absolute SHAP value.

In [ ]:
# Aggregate across instances
summary_stability = stability_df.groupby('Feature').agg({
    'Mean_SHAP': 'mean',
    'Abs_Mean_SHAP': 'mean',
    'Std_SHAP': 'mean'
}).reset_index()

summary_stability.columns = ['Feature', 'Mean_SHAP', 'Abs_Mean_SHAP', 'Mean_Std_SHAP']
summary_stability['Instability_Threshold'] = 0.1 * summary_stability['Abs_Mean_SHAP']
summary_stability['Is_Unstable'] = summary_stability['Mean_Std_SHAP'] > summary_stability['Instability_Threshold']

summary_stability_sorted = summary_stability.sort_values('Mean_Std_SHAP', ascending=False)

print("Explanation Stability Summary Table")
print("=" * 100)
print(summary_stability_sorted[['Feature', 'Mean_SHAP', 'Abs_Mean_SHAP', 'Mean_Std_SHAP', 'Is_Unstable']].to_string(index=False))
print("=" * 100)

n_unstable = summary_stability['Is_Unstable'].sum()
print(f"\nTotal unstable features (Std Dev > 10% of Mean Abs SHAP): {n_unstable} out of {len(summary_stability)}")
print(f"Unstable features: {summary_stability[summary_stability['Is_Unstable']]['Feature'].tolist()}")

### Expected Outputs for Step 4

A table with columns: Feature, Mean_SHAP, Abs_Mean_SHAP, Mean_Std_SHAP, Is_Unstable. Any feature flagged as Is_Unstable equals True has a standard deviation exceeding 10% of its mean absolute attribution, indicating fragility. The count and list of unstable features should be clearly reported.

## Questions

1. For the tabular model, which three features had the largest mean absolute SHAP values in the global summary across 50 test instances? Interpret what these features mean in the context of this binary classification task (they play the role that credit history features would play in a lending model).

2. In the LIME image explanations, did superpixels supporting the predicted class have positive or negative coefficients? If an explanation coefficient is large and negative for a superpixel, what does that tell you about that region's influence on the model's decision?

3. In the perturbation stability analysis, was any feature flagged as unstable with standard deviation greater than 10% of mean absolute SHAP? If so, what does this instability imply about whether those features are reliable for auditing model decisions?

## Optional Challenges

1. **Integrated Gradients Implementation** (worked example in the cells below): Implement Integrated Gradients from scratch for the CIFAR-10 image classifier using the captum library or manual gradient computation. Generate pixel-level attribution maps for one test image and compare attributions to LIME for the same image. Report whether the two methods highlight the same or different image regions.

2. **Explanation Manipulation Attack**: Using the approach from Slack et al., add small adversarial perturbations to one of the test instances to flip which features LIME identifies as most important, while keeping the model's prediction unchanged. Report the original and manipulated LIME attributions and the maximum perturbation magnitude needed to flip top-feature rankings.

These challenges are not required for lab completion and assume familiarity with gradient-based attacks and adversarial perturbations.

## Optional Challenge 1 (Worked): Integrated Gradients Attribution Maps

The cells below implement the first optional challenge: pixel-level attribution maps via Integrated Gradients, a completeness check, and a side-by-side comparison with LIME on the same image. Integrated Gradients integrates the model's gradients along a straight-line path from a black (all-zeros) baseline image to the query image, producing one attribution value per pixel and color channel.


In [ ]:
# Integrated Gradients with captum
from captum.attr import IntegratedGradients

ig = IntegratedGradients(model_img)

# Explain the first test image (same image LIME explained above)
img_idx = 0
input_tensor = images[img_idx:img_idx + 1].to(device)      # NCHW, values in [0, 1]
baseline = torch.zeros_like(input_tensor)                  # all-zeros (black) image
target = int(pred_classes[img_idx])                        # predicted class

attributions = ig.attribute(
    input_tensor,
    baselines=baseline,
    target=target,
    n_steps=100,                                           # numerical integration steps
)

attr = attributions.squeeze(0).cpu().numpy()               # (C, H, W)
attr_map = attr.sum(axis=0)                                # (H, W) pixel-level map

# Completeness check: sum of attributions ~ f(X) - f(baseline)
baseline_prob = predict_fn(np.zeros_like(images_np[img_idx:img_idx + 1]))[0, target]
input_prob = preds[img_idx, target]
print(f"Predicted class: {class_names[target]}")
print(f"f(input) - f(baseline)      = {input_prob - baseline_prob:.4f}")
print(f"Sum of pixel attributions   = {attr_map.sum():.4f}")

In [ ]:
# Visualize the Integrated Gradients attribution map next to LIME
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 1. Original image
axes[0].imshow(images_np[img_idx])
axes[0].set_title(f"Original\n(true: {class_names[true_labels[img_idx]]}, pred: {class_names[target]})")

# 2. Attribution heatmap (red = pushes toward the class, blue = pushes away)
vmax = np.abs(attr_map).max()
im = axes[1].imshow(attr_map, cmap="seismic", vmin=-vmax, vmax=vmax)
axes[1].set_title("Integrated Gradients\nattribution map")
fig.colorbar(im, ax=axes[1], fraction=0.046)

# 3. Heatmap overlaid on the image
axes[2].imshow(images_np[img_idx])
axes[2].imshow(attr_map, cmap="seismic", vmin=-vmax, vmax=vmax, alpha=0.55)
axes[2].set_title("IG overlay")

# 4. LIME top-superpixel mask for the same image (from the LIME step above)
temp_img, mask = explanation.get_image_and_mask(
    top_label, positive_only=False, num_features=10, hide_rest=False
)
axes[3].imshow(mark_boundaries(temp_img, mask))
axes[3].set_title("LIME top-10 superpixels\n(same image)")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: do IG and LIME highlight the same regions?
# Aggregate IG pixel attributions within each LIME superpixel and correlate
# with the LIME coefficients (Spearman rank correlation).
from scipy.stats import spearmanr

segments = explanation.segments  # superpixel id per pixel
lime_ids, lime_w, ig_w = [], [], []
for sp_id, w in explanation.local_exp[top_label]:
    lime_ids.append(sp_id)
    lime_w.append(w)
    ig_w.append(attr_map[segments == sp_id].mean())

rho, pval = spearmanr(lime_w, ig_w)
print(f"Spearman rank correlation between LIME coefficients and mean IG")
print(f"attribution per superpixel: rho = {rho:.3f} (p = {pval:.3g})")
print()
print("Interpretation: rho near 1 means the two methods rank image regions")
print("similarly; rho near 0 means they highlight different regions.")
